# Day 2 - 세션 3 : Gradio로 웹 앱 만들기

## 📌 지금까지 만든 것의 한계

세션 1~2에서 우리는:
- PyCaret으로 콘크리트 압축강도를 예측하는 **최적 모델**을 찾았고 (`compare_models`)
- 그 모델이 **왜** 그렇게 예측하는지 SHAP 그래프로 확인했고 (`interpret_model`)
- 하이퍼파라미터 튜닝으로 성능을 살짝 더 끌어올렸습니다 (`tune_model`)

하지만 이 모든 작업은 **Jupyter Notebook 안에서만** 작동합니다.
다른 사람이 이 모델을 쓰려면 Python 코드를 직접 실행해야 하는데, 비개발자인 현장 담당자에게는 불가능한 일입니다.

## 🎯 세션 3의 목표

> 배합 수치를 입력창에 넣으면 → 압축강도 예측값이 바로 나오는 **웹 앱**을 만든다

코드를 전혀 모르는 품질팀 담당자도 사용할 수 있고, 발표 자리에서 바로 시연할 수 있는 결과물을 만듭니다.

## ✅ 세션 3 목표

1. **Gradio**가 무엇이고 왜 쓰는지 이해한다
2. 입력값을 받아 모델 예측을 돌려주는 **예측 함수**를 직접 작성한다
3. 이 함수를 **웹 인터페이스(Gradio)**로 감싸서 작동하는 앱을 완성한다


## STEP 1. 모델 준비하기

새 노트북이므로, 이전 세션에서 했던 과정을 다시 실행해서 **튜닝된 최종 모델**을 준비합니다.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()

from pycaret.regression import *

s = setup(data=df, target='concrete_compressive_strength', session_id=123)
best_model = compare_models()
final_model = tune_model(best_model, n_iter=30, optimize='R2')

print("최종 모델:", final_model)

,Description,Value
0,Session id,123
1,Target,concrete_compressive_strength
2,Target type,Regression
3,Original data shape,"(1030, 9)"
4,Transformed data shape,"(1030, 9)"
5,Transformed train set shape,"(721, 9)"
6,Transformed test set shape,"(309, 9)"
7,Numeric features,8
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,3.2006,21.9523,4.6356,0.9215,0.1511,0.1106,0.1780
et,Extra Trees Regressor,3.2530,23.4571,4.7985,0.9166,0.1527,0.1128,0.0160
gbr,Gradient Boosting Regressor,3.7650,27.6610,5.2383,0.9007,0.1634,0.1295,0.0110
rf,Random Forest Regressor,3.6619,28.2038,5.2328,0.8993,0.1679,0.1290,0.0230
dt,Decision Tree Regressor,4.5467,43.9027,6.5348,0.8442,0.2165,0.1580,0.0030
ada,AdaBoost Regressor,6.1832,57.2779,7.5252,0.7944,0.2782,0.2605,0.0110
knn,K Neighbors Regressor,7.4127,94.3747,9.6708,0.6625,0.3208,0.2894,0.0050
ridge,Ridge Regression,8.3627,113.1336,10.6141,0.5892,0.3398,0.3216,0.0030
lar,Least Angle Regression,8.3627,113.1337,10.6141,0.5892,0.3398,0.3216,0.0030
lr,Linear Regression,8.3627,113.1337,10.6141,0.5892,0.3398,0.3216,0.2610


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,2.8208,18.3020,4.2781,0.9279,0.1113,0.0855
1,3.6682,24.8109,4.9811,0.9269,0.1533,0.1193
2,3.0836,16.3088,4.0384,0.9347,0.1572,0.1077
3,3.2191,18.0475,4.2482,0.9485,0.1645,0.1218
4,3.1705,21.1350,4.5973,0.9301,0.1771,0.1249
5,3.7522,28.9140,5.3772,0.9059,0.1841,0.1354
6,3.3142,21.0186,4.5846,0.9207,0.1565,0.1103
7,2.9963,16.9596,4.1182,0.9345,0.1602,0.1168
8,2.7919,12.9184,3.5942,0.9382,0.1095,0.0893


Fitting 10 folds for each of 30 candidates, totalling 300 fits
최종 모델: LGBMRegressor(bagging_fraction=1.0, bagging_freq=4, feature_fraction=0.5,
              learning_rate=0.15, min_child_samples=46, min_split_gain=0.3,
              n_estimators=280, n_jobs=-1, num_leaves=100, random_state=123,
              reg_alpha=4, reg_lambda=3)


## STEP 2. Gradio란 무엇인가

**Gradio**는 머신러닝 모델을 **웹 인터페이스로 감싸주는** 파이썬 라이브러리입니다.
입력창(숫자 입력, 슬라이더 등)과 출력창을 자동으로 만들어주는, **모델과 사용자 사이의 번역기** 역할을 합니다.

### 작동 흐름

```
사용자가 배합값 입력 (Cement, Water, Age 등)
            ↓
Gradio가 입력값을 모델에 전달
            ↓
PyCaret 모델이 압축강도 예측
            ↓
Gradio가 결과를 화면에 표시
```

### Gradio 앱을 만드는 데 필요한 3가지

1. **입력 컴포넌트 (Inputs)** - `gr.Number()`, `gr.Slider()` 등으로 만든 입력창
2. **예측 함수 (Function)** - 입력값을 받아 결과를 반환하는 파이썬 함수 1개
3. **출력 컴포넌트 (Outputs)** - 결과를 보여줄 출력창

이 세 가지를 `gr.Interface(fn=함수, inputs=[...], outputs=[...])`로 묶으면 앱이 완성됩니다.


## STEP 3. 예측 함수 만들기

Gradio 앱의 핵심은 **예측 함수**입니다. 이 함수는:

1. 사용자가 입력한 8개 숫자(시멘트, 슬래그, 플라이애시, 물, 감수제, 굵은골재, 잔골재, 양생일)를 받아서
2. PyCaret이 이해할 수 있는 **표(DataFrame) 형태**로 만들고
3. `predict_model()`로 예측한 뒤
4. 예측된 압축강도(MPa) 값을 돌려줍니다

`predict_model(model, data=새로운_데이터)`를 호출하면, 결과 표에 **`prediction_label`**이라는 컬럼이 추가되어 예측값이 담깁니다.


In [2]:
def predict_strength(cement, blast_furnace_slag, fly_ash, water,
                      superplasticizer, coarse_aggregate, fine_aggregate, age):
    # 1. 입력값을 모델이 이해하는 표(DataFrame) 형태로 변환
    input_df = pd.DataFrame([{
        'cement': cement,
        'blast_furnace_slag': blast_furnace_slag,
        'fly_ash': fly_ash,
        'water': water,
        'superplasticizer': superplasticizer,
        'coarse_aggregate': coarse_aggregate,
        'fine_aggregate': fine_aggregate,
        'age': age,
    }])

    # 2. 모델로 예측
    result = predict_model(final_model, data=input_df)

    # 3. 예측된 압축강도(MPa) 반환
    return round(result['prediction_label'].iloc[0], 2)

## STEP 4. 함수 테스트하기

Gradio로 감싸기 전에, 함수가 잘 작동하는지 직접 호출해서 확인합니다.
데이터셋의 첫 번째 행 값을 그대로 넣어서, 예측값이 실제값과 비슷한지 비교해봅시다.


In [3]:
sample = df.iloc[0]
print("입력값:")
print(sample.drop('concrete_compressive_strength'))

pred = predict_strength(
    cement=sample['cement'],
    blast_furnace_slag=sample['blast_furnace_slag'],
    fly_ash=sample['fly_ash'],
    water=sample['water'],
    superplasticizer=sample['superplasticizer'],
    coarse_aggregate=sample['coarse_aggregate'],
    fine_aggregate=sample['fine_aggregate'],
    age=sample['age'],
)

print(f"\n예측 압축강도: {pred} MPa")
print(f"실제 압축강도: {sample['concrete_compressive_strength']} MPa")

입력값:
cement                 540.0
blast_furnace_slag       0.0
fly_ash                  0.0
water                  162.0
superplasticizer         2.5
coarse_aggregate      1040.0
fine_aggregate         676.0
age                     28.0
Name: 0, dtype: float64


[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.5, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.5
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0

예측 압축강도: 72.89 MPa
실제 압축강도: 79.99 MPa


## STEP 5. Gradio 인터페이스 만들기

이제 `predict_strength` 함수를 Gradio로 감싸봅니다.

- **입력 컴포넌트**: 8개 변수에 대해 `gr.Number()`를 사용합니다. `label`로 한글 이름을, `value`로 기본값(데이터셋 평균)을 지정합니다.
- **출력 컴포넌트**: 예측 압축강도를 보여줄 `gr.Number()` 1개

`demo.launch()`를 실행하면 로컬 웹 서버가 켜지고, 노트북 안에 인터페이스가 표시됩니다.
(Colab에서는 `share=True`를 추가하면 외부에서도 접속 가능한 공개 URL이 함께 발급됩니다.)


In [4]:
import gradio as gr

# 입력 컴포넌트: 데이터셋 평균값을 기본값으로 사용
means = df.drop(columns=['concrete_compressive_strength']).mean()

inputs = [
    gr.Number(label="시멘트 (cement, kg/m³)", value=round(means['cement'], 1)),
    gr.Number(label="고로 슬래그 (blast_furnace_slag, kg/m³)", value=round(means['blast_furnace_slag'], 1)),
    gr.Number(label="플라이애시 (fly_ash, kg/m³)", value=round(means['fly_ash'], 1)),
    gr.Number(label="물 (water, kg/m³)", value=round(means['water'], 1)),
    gr.Number(label="고성능 감수제 (superplasticizer, kg/m³)", value=round(means['superplasticizer'], 1)),
    gr.Number(label="굵은 골재 (coarse_aggregate, kg/m³)", value=round(means['coarse_aggregate'], 1)),
    gr.Number(label="잔골재 (fine_aggregate, kg/m³)", value=round(means['fine_aggregate'], 1)),
    gr.Number(label="양생 일수 (age, day)", value=round(means['age'], 1)),
]

output = gr.Number(label="예측 압축강도 (MPa)")

demo = gr.Interface(
    fn=predict_strength,
    inputs=inputs,
    outputs=output,
    title="콘크리트 압축강도 예측기",
    description="배합 비율과 양생 일수를 입력하면, 28일 압축강도를 예측합니다.",
)

demo.launch(prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## 📌 실행 결과 확인하기

위 셀을 실행하면 **로컬 주소(예: http://127.0.0.1:7860)**가 출력되고, 노트북 안에 입력창들이 표시됩니다.

- 입력값을 바꿔보고 "Submit" 버튼을 누르면, 예측 압축강도가 즉시 갱신됩니다.
- Colab에서 실행 중이라면, `demo.launch(share=True)`로 바꾸면 `https://xxxx.gradio.app` 형태의 **공개 URL**이 추가로 발급됩니다.
  이 URL을 카카오톡 등으로 보내면, 코드 설치 없이 누구나 휴대폰/PC 브라우저로 접속해서 사용할 수 있습니다.

다음 셀에서는 실행 중인 서버를 종료합니다 (노트북을 계속 사용하려면 정리가 필요합니다).


In [5]:
demo.close()

Closing server running on port: 7860


## STEP 6. 실습 - KS 기준 충족 여부 표시 추가하기

지금 만든 앱은 예측 압축강도(MPa) 숫자만 보여줍니다. 여기에 한 가지를 추가해봅시다.

> 예측값이 **KS L 5201 기준(42.5 MPa) 이상이면 "충족 ✅"**, 미만이면 **"미달 ❌"**을 함께 보여주기

이를 위해:
1. 예측 함수가 **(예측값, 충족 여부 문자열)** 두 가지를 반환하도록 수정합니다
2. `gr.Interface`의 `outputs`에 `gr.Number()`와 `gr.Label()` 두 개를 지정합니다


In [6]:
KS_STANDARD = 42.5

def predict_strength_with_check(cement, blast_furnace_slag, fly_ash, water,
                                 superplasticizer, coarse_aggregate, fine_aggregate, age):
    input_df = pd.DataFrame([{
        'cement': cement,
        'blast_furnace_slag': blast_furnace_slag,
        'fly_ash': fly_ash,
        'water': water,
        'superplasticizer': superplasticizer,
        'coarse_aggregate': coarse_aggregate,
        'fine_aggregate': fine_aggregate,
        'age': age,
    }])

    result = predict_model(final_model, data=input_df)
    pred = round(result['prediction_label'].iloc[0], 2)

    if pred >= KS_STANDARD:
        status = f"KS L 5201 기준 충족 ✅ ({KS_STANDARD} MPa 이상)"
    else:
        status = f"KS L 5201 기준 미달 ❌ ({KS_STANDARD} MPa 미만)"

    return pred, status


demo2 = gr.Interface(
    fn=predict_strength_with_check,
    inputs=inputs,
    outputs=[
        gr.Number(label="예측 압축강도 (MPa)"),
        gr.Label(label="KS L 5201 (42.5 MPa) 기준 충족 여부"),
    ],
    title="콘크리트 압축강도 예측기 (KS 기준 포함)",
    description="배합 비율과 양생 일수를 입력하면, 28일 압축강도와 KS 기준 충족 여부를 함께 보여줍니다.",
)

demo2.launch(prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [7]:
demo2.close()

Closing server running on port: 7860


## 📝 Day 2 전체 정리

```
UCI Concrete 데이터 (1,030샘플 × 9컬럼)
            ↓
PyCaret AutoML — 10+ 모델 자동 비교        (세션 1)
            ↓
Top 3 모델 선정 (R²·RMSE·학습시간 기준)     (세션 1)
            ↓
interpret_model() — 변수 중요도 시각화      (세션 2)
            ↓
tune_model() — 하이퍼파라미터 자동 튜닝     (세션 2)
            ↓
Gradio — 누구나 쓸 수 있는 웹 앱            (세션 3) ✅
```

### ✅ Day 2 전체 산출물

- PyCaret Top 3 모델 비교표
- 변수 중요도(SHAP) 그래프 + 튜닝된 모델
- 배합값을 입력하면 압축강도를 예측해주는 **작동하는 웹 앱**

### 🚀 한 걸음 더 나아간다면

- 입력 변수를 8개 모두 입력하는 게 번거롭다면, 자주 쓰는 배합 몇 가지를 **프리셋 버튼**으로 만들 수 있습니다
- `demo.launch(share=True)`로 외부에 공개해 동료들과 함께 테스트해볼 수 있습니다
- 예측이 크게 빗나가는 입력 구간(예: `age`가 매우 작거나 큰 경우)이 있는지 직접 탐색해보면, 모델의 한계를 더 잘 이해할 수 있습니다
